# Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import logging
logging.disable(logging.CRITICAL)

In [3]:
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime

In [4]:
import os
import json

In [5]:
# --- F1 Data ---
import fastf1  # needed for qualifying data and 2026 season data 
# Jolpica API (no library needed, just requests)
# Base URL: https://api.jolpi.ca/ergast/f1/
#gives data for historical F1 seasons completed

# Helper Functions

In [6]:
# --- Helper function for Jolpica ---
def get_jolpica_results(year, driver_code):
    url = f"https://api.jolpi.ca/ergast/f1/{year}/drivers/{driver_code}/results/?limit=100"
    response = requests.get(url)
    data = response.json()
    races = data['MRData']['RaceTable']['Races']
    results = []
    for race in races:
        result = race['Results'][0]
        results.append({
            'Driver': driver_code,
            'Year': year,
            'Round': int(race['round']),
            'EventName': race['raceName'],
            'RookiePosition': float(result['position']),
            'GridPosition': float(result['grid']),
            'RookiePoints': float(result['points']),
            'Status': result['status']
        })
    return pd.DataFrame(results)

In [7]:
def get_teammate(year, driver_code):
    url = f"https://api.jolpi.ca/ergast/f1/{year}/constructors.json?limit=100"
    response = requests.get(url)
    constructors = response.json()['MRData']['ConstructorTable']['Constructors']
    
    # Find driver's constructor
    driver_url = f"https://api.jolpi.ca/ergast/f1/{year}/drivers/{driver_code}/constructors.json"
    response = requests.get(driver_url)
    constructor = response.json()['MRData']['ConstructorTable']['Constructors'][0]['constructorId']
    
    # Get all drivers for that constructor that season
    team_url = f"https://api.jolpi.ca/ergast/f1/{year}/constructors/{constructor}/drivers.json"
    response = requests.get(team_url)
    team_drivers = response.json()['MRData']['DriverTable']['Drivers']
    
    # Return the other driver
    teammates = [d['driverId'] for d in team_drivers if d['driverId'] != driver_code]
    return teammates[0] if teammates else None

In [8]:
def save_all(data_dict, path='Analysis/Data'):
    os.makedirs(path, exist_ok=True)
    for name, obj in data_dict.items():
        if isinstance(obj, pd.DataFrame):
            obj.to_csv(f'{path}/{name}.csv', index=False)
        else:
            with open(f'{path}/{name}.json', 'w') as f:
                json.dump(obj, f)
    print(f"Saved {len(data_dict)} files.")

# Defining Constants

In [9]:
# Define rookies we're going to use for comparison later along with their debut seasons
drivers = {
    'Hamilton':   (2007, 'hamilton'),
    'Verstappen': (2015, 'max_verstappen'),
    'Leclerc':    (2018, 'leclerc'),
    'Norris':     (2019, 'norris'),
}

In [10]:
rookies_2025 = {
    'Antonelli': ('antonelli', 'ANT'),
    'Hadjar':    ('hadjar',    'HAD'),
    'Bortoleto': ('bortoleto', 'BOR'),
    'Bearman':   ('bearman',   'BEA')
}

# Pulling data

In [11]:
# ============================================================
# STEP 1 — RACE SCHEDULES
# ============================================================

schedule_2025 = fastf1.get_event_schedule(2025)
schedule_2026 = fastf1.get_event_schedule(2026)

races_2025 = schedule_2025[schedule_2025['RoundNumber'] > 0].copy()
races_2025['Season'] = 2025

completed_2026 = schedule_2026[
    (schedule_2026['RoundNumber'] > 0) &
    (schedule_2026['EventDate'] < datetime.today())
].copy()
completed_2026['Season'] = 2026

all_races = pd.concat([races_2025, completed_2026], ignore_index=True)

# Build schedule lookup for dates
all_schedules = pd.concat([
    schedule_2025[['EventName', 'EventDate']].assign(Season=2025),
    schedule_2026[['EventName', 'EventDate']].assign(Season=2026)
], ignore_index=True)
all_schedules['EventDate'] = pd.to_datetime(all_schedules['EventDate'])

print(f"Total races: {len(all_races)}")


Total races: 27


In [12]:
# ============================================================
# STEP 2 — MERCEDES 2025-2026 (FastF1)
# ============================================================

merc_list = []

for _, race in all_races.iterrows():
    try:
        session = fastf1.get_session(race['Season'], race['RoundNumber'], 'R')
        session.load(telemetry=False, weather=False, messages=False)

        drivers_data = session.results[
            session.results['Abbreviation'].isin(['ANT', 'RUS'])
        ][['Abbreviation', 'Position', 'GridPosition', 'Status', 'Points']].copy()

        drivers_data['EventName'] = race['EventName']
        drivers_data['Season'] = race['Season']
        merc_list.append(drivers_data)

    except Exception as e:
        print(f"✗ {race['Season']} {race['EventName']}: {e}")

merc_results = pd.concat(merc_list, ignore_index=True)
merc_results = merc_results.merge(
    all_schedules[['EventName', 'Season', 'EventDate']],
    on=['EventName', 'Season'],
    how='left'
)
merc_results['Driver'] = merc_results['Abbreviation'].map({'ANT': 'Antonelli', 'RUS': 'Russell'})

print(f"Mercedes results: {len(merc_results)} entries")

Mercedes results: 54 entries


In [13]:
# ============================================================
# STEP 3 — GENERATIONAL TALENTS DEBUT SEASONS (Jolpica)
# ============================================================

rookie_list = []
for driver_name, (year, code) in drivers.items():
    df = get_jolpica_results(year, code)
    df['Driver'] = driver_name
    rookie_list.append(df)
    print(f"✓ {driver_name}")

rookie_df = pd.concat(rookie_list, ignore_index=True)
rookie_df = rookie_df.rename(columns={'RookiePosition': 'Position', 'GridPosition': 'Grid'})

# Auto-pull teammates
teammate_list = []
teammate_names = {}
for driver_name, (year, code) in drivers.items():
    teammate_code = get_teammate(year, code)
    teammate_names[driver_name] = teammate_code.replace('_', ' ').title()
    df = get_jolpica_results(year, teammate_code)
    df['Driver'] = driver_name
    df = df.rename(columns={'RookiePosition': 'TeammatePosition', 'GridPosition': 'TeammateGrid'})
    teammate_list.append(df)
    print(f"✓ {driver_name} teammate: {teammate_code}")

teammate_df = pd.concat(teammate_list, ignore_index=True)

# Rookie years lookup
rookie_years = {driver_name: year for driver_name, (year, code) in drivers.items()}

✓ Hamilton
✓ Verstappen
✓ Leclerc
✓ Norris
✓ Hamilton teammate: alonso
✓ Verstappen teammate: sainz
✓ Leclerc teammate: ericsson
✓ Norris teammate: sainz


In [14]:
# ============================================================
# STEP 4 — 2025 ROOKIE CLASS (Jolpica + FastF1)
# ============================================================

# Jolpica for rookie results
rookie_results_2025_list = []
for name, (code, abbr) in rookies_2025.items():
    df = get_jolpica_results(2025, code)
    df['Driver'] = name
    rookie_results_2025_list.append(df)
    print(f"✓ {name}")

rookie_results_2025 = pd.concat(rookie_results_2025_list, ignore_index=True)
rookie_results_2025 = rookie_results_2025.rename(columns={
    'RookiePosition': 'Position',
    'GridPosition': 'Grid'
})

# FastF1 — find primary teammate (most races) for each rookie
teammate_counts = {name: {} for name in rookies_2025}

for _, race in races_2025.iterrows():
    try:
        session = fastf1.get_session(2025, race['RoundNumber'], 'R')
        session.load(telemetry=False, weather=False, messages=False)

        for name, (code, abbr) in rookies_2025.items():
            rookie_row = session.results[session.results['Abbreviation'] == abbr]
            if rookie_row.empty:
                continue

            team = rookie_row['TeamName'].values[0]
            teammate = session.results[
                (session.results['TeamName'] == team) &
                (session.results['Abbreviation'] != abbr)
            ]

            if not teammate.empty:
                tm_abbr = teammate['Abbreviation'].values[0]
                teammate_counts[name][tm_abbr] = teammate_counts[name].get(tm_abbr, 0) + 1

    except Exception as e:
        print(f"✗ {race['EventName']}: {e}")

# Pick primary teammate per rookie
primary_teammates = {
    name: max(counts, key=counts.get)
    for name, counts in teammate_counts.items()
    if counts
}

print("\nPrimary teammates:")
for name, tm in primary_teammates.items():
    print(f"  {name}: {tm} ({teammate_counts[name]})")

# FastF1 — pull primary teammate results only
teammate_fastf1_list = []

for _, race in races_2025.iterrows():
    try:
        session = fastf1.get_session(2025, race['RoundNumber'], 'R')
        session.load(telemetry=False, weather=False, messages=False)

        for name, tm_abbr in primary_teammates.items():
            tm_row = session.results[session.results['Abbreviation'] == tm_abbr]
            if tm_row.empty:
                continue

            tm = tm_row[['Abbreviation', 'Position', 'GridPosition', 'Status']].copy()
            tm['Driver'] = name
            tm['EventName'] = race['EventName']
            tm['Round'] = race['RoundNumber']
            tm['Year'] = 2025
            teammate_fastf1_list.append(tm)

    except Exception as e:
        print(f"✗ {race['EventName']}: {e}")

teammate_fastf1_df = pd.concat(teammate_fastf1_list, ignore_index=True)
teammate_fastf1_df = teammate_fastf1_df.rename(columns={
    'Position': 'TeammatePosition',
    'GridPosition': 'TeammateGrid',
    'Abbreviation': 'TeammateAbbr'
})

# Save primary teammates for reference
primary_teammates_named = {
    name: tm_abbr for name, tm_abbr in primary_teammates.items()
}


print(f"\n2025 rookie results: {len(rookie_results_2025)} entries")
print(f"2025 rookie teammates: {len(teammate_fastf1_df)} entries")

✓ Antonelli
✓ Hadjar
✓ Bortoleto
✓ Bearman

Primary teammates:
  Antonelli: RUS ({'RUS': 24})
  Hadjar: LAW ({'TSU': 2, 'LAW': 22})
  Bortoleto: HUL ({'HUL': 24})
  Bearman: OCO ({'OCO': 24})

2025 rookie results: 96 entries
2025 rookie teammates: 96 entries


# Save Data

In [15]:
# Driver metadata — all reference info in one file
driver_metadata = {
    'drivers':             drivers,           # name -> (year, jolpica code)
    'teammate_names':      teammate_names,     # name -> teammate name
    'rookie_years':        rookie_years,       # name -> debut year
    'rookies_2025':        rookies_2025,       # name -> (jolpica code, abbr)
    'primary_teammates':   primary_teammates   # name -> primary teammate abbr
}

save_all({
    # All reference/metadata in one file
    'driver_metadata':     driver_metadata,
    
    # FastF1 — Antonelli and Russell race + qualifying results 2025-2026
    'merc_results':        merc_results,
    
    # Jolpica — debut season results for generational talents + their teammates
    'rookie_df_gen_talent':           rookie_df,
    'teammate_df_gen_talent':         teammate_df,
    
    # FastF1/Jolpica — 2025 rookie class results + their primary teammates
    'rookie_df_2025': rookie_results_2025,
    'teammate_df_2025':  teammate_fastf1_df,
})

Saved 6 files.
